# Specialist (Biner) → Generalist (10 Head) Multi-Task Learning + CI 95% + Expand (Multi-MNIST)

Alur:
1. **10 spesialis** one-vs-all. Tiap spesialis punya output 2 kelas (*bukan kelas dia* / *kelas dia*), dilatih dengan **label asli** MNIST. Bobot loss: **0.9** untuk "kelas dia", **0.1** untuk "bukan kelas dia".
2. **Generalis** dengan **10 head** (1 head = 1 kelas, multi-label: ada/tidak ada di tumpukan gambar). Generalis **tidak memakai label asli**, hanya soft target dari spesialis.
3. Output head dianggap **ada** jika di atas ekspektasi `1/10` (tiap task).
4. **Confidence interval 95%** (MC-dropout + random shift): kelas dianggap ada jika **batas bawah CI 95% > 1/10**.
5. **Expand**: tambah sampel (CI makin rapat), hitung ulang jumlah kelas. Jika jumlah berubah (mis. 4 → 3) expand lagi; jika sama dengan ronde sebelumnya, berhenti dan itulah jumlah kelas di gambar.

## 1. Setup & Konfigurasi

In [ ]:
import math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets, models

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', dev)

CFG = dict(
    N_TRAIN_MULTI = 20000,
    N_VAL_MULTI   = 2000,
    N_TEST_MULTI  = 5000,
    N_SPEC_HARD   = 20000,     # crop dari tumpukan untuk melatih spesialis
    BS            = 128,
    LR            = 1e-3,
    SPEC_EPOCHS   = 3,
    GEN_EPOCHS    = 6,
    GEN_ARCH      = 'resnet50',
    W_NEG         = 0.1,       # bobot "bukan kelas dia"
    W_POS         = 0.9,       # bobot "kelas dia"
    PRIOR_CORRECT = True,      # koreksi bias akibat bobot 0.9/0.1 saat inferensi spesialis
    METHOD        = 'ls',      # 'ls' | 'pcgrad' | 'mgda' | 'famo'
    DROPOUT       = 0.2,       # dipakai sebagai MC-dropout
    CI_Z          = 1.96,      # 95%
    N0            = 8,         # jumlah sampel awal
    MAX_ROUNDS    = 4,         # batas ronde expand (sampel digandakan tiap ronde)
    SEED          = 0,
)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
set_seed(CFG['SEED'])

## 2. Dataset Multi-MNIST (2–4 digit bertumpuk)

In [ ]:
CANVAS, DS = 56, 28           # kanvas 56x56, digit tetap 28x28 asli
MAX_OVERLAP = 0.25            # maks. overlap kotak antar digit (fraksi luas 28x28)
K_MIN, K_MAX = 2, 4
CLASS_NAMES = tuple(str(d) for d in range(10))   # kelas = digit 0-9
N_CLASSES = len(CLASS_NAMES)
THR = 1.0 / N_CLASSES          # ekspektasi per task = 0.1

mnist_tr = datasets.MNIST('./data', train=True, download=True)
mnist_te = datasets.MNIST('./data', train=False, download=True)
all_tr_x, all_tr_y = mnist_tr.data.numpy(), mnist_tr.targets.numpy()
mn_te_x, mn_te_y = mnist_te.data.numpy(), mnist_te.targets.numpy()

perm = np.random.RandomState(CFG['SEED']).permutation(len(all_tr_x))
mn_tr_x, mn_tr_y = all_tr_x[perm[:54000]], all_tr_y[perm[:54000]]
mn_val_x, mn_val_y = all_tr_x[perm[54000:]], all_tr_y[perm[54000:]]

def make_multi_flexible(imgs, labels, n, k_min=K_MIN, k_max=K_MAX, seed=1,
                        max_overlap=MAX_OVERLAP, n_tries=30):
    rng = np.random.RandomState(seed)
    X = np.zeros((n, CANVAS, CANVAS), dtype=np.uint8)
    Y = np.full((n, k_max), -1, dtype=np.int64)
    P = np.full((n, k_max, 2), -1, dtype=np.int64)
    off = CANVAS - DS
    for i in range(n):
        k_items = rng.randint(k_min, k_max + 1)
        idx = rng.randint(0, len(imgs), size=k_items)
        boxes = []
        for k, j in enumerate(idx):
            best, best_ov = None, 9.0
            for _ in range(n_tries):          # cari posisi dengan overlap kecil
                py, px = rng.randint(0, off + 1), rng.randint(0, off + 1)
                ov = max([max(0, DS - abs(py - by)) * max(0, DS - abs(px - bx)) / (DS * DS)
                          for by, bx in boxes], default=0.0)
                if ov < best_ov:
                    best, best_ov = (py, px), ov
                if ov <= max_overlap:
                    break
            py, px = best
            boxes.append((py, px))
            patch = X[i, py:py+DS, px:px+DS].astype(np.uint16) + imgs[j].astype(np.uint16)
            X[i, py:py+DS, px:px+DS] = np.minimum(patch, 255).astype(np.uint8)
            Y[i, k] = labels[j]
            P[i, k] = [py, px]
    return X, Y, P

def mean_max_overlap(P):
    # statistik: rata-rata overlap terbesar per gambar (fraksi luas digit)
    vals = []
    for p in P:
        b = p[p[:, 0] != -1]
        ov = [max(0, DS - abs(b[a, 0] - b[c, 0])) * max(0, DS - abs(b[a, 1] - b[c, 1])) / (DS * DS)
              for a in range(len(b)) for c in range(a + 1, len(b))]
        vals.append(max(ov) if ov else 0.0)
    return float(np.mean(vals))

def crop_pad(X, P, k):
    out = np.zeros_like(X)
    o = (CANVAS - DS) // 2
    for i in range(len(X)):
        if P[i, k, 0] != -1:
            y, x = P[i, k]
            out[i, o:o+DS, o:o+DS] = X[i, y:y+DS, x:x+DS]
    return out

def pad_mnist(imgs):
    out = np.zeros((len(imgs), CANVAS, CANVAS), np.uint8)
    o = (CANVAS - DS) // 2
    out[:, o:o+DS, o:o+DS] = imgs
    return out

def multihot(Y):
    # Y: (n, K_MAX) berisi id kelas atau -1  ->  (n, 10) bool: kelas apa saja yang ada
    m = np.zeros((len(Y), N_CLASSES), dtype=bool)
    for k in range(Y.shape[1]):
        v = np.where(Y[:, k] != -1)[0]
        m[v, Y[v, k]] = True
    return m

# Tumpukan gambar (train/val/test)
Xg_tr,  Yg_tr,  Pg_tr  = make_multi_flexible(mn_tr_x,  mn_tr_y,  CFG['N_TRAIN_MULTI'], seed=1)
Xg_val, Yg_val, Pg_val = make_multi_flexible(mn_val_x, mn_val_y, CFG['N_VAL_MULTI'],   seed=2)
Xg_te,  Yg_te,  Pg_te  = make_multi_flexible(mn_te_x,  mn_te_y,  CFG['N_TEST_MULTI'],  seed=3)
Mh_val, Mh_te = multihot(Yg_val), multihot(Yg_te)   # ground truth hanya untuk EVALUASI

# Data spesialis: gambar tunggal bersih + crop dari tumpukan, semuanya berlabel ASLI
n_h = (CFG['N_SPEC_HARD'] + K_MAX - 1) // K_MAX
Xh_raw, Yh_raw, Ph_raw = make_multi_flexible(mn_tr_x, mn_tr_y, n_h, seed=4)
Xs_h, Ys_h = [], []
for k in range(K_MAX):
    v = Ph_raw[:, k, 0] != -1
    Xs_h.append(crop_pad(Xh_raw, Ph_raw, k)[v]); Ys_h.append(Yh_raw[v, k])
Xs_h = np.concatenate(Xs_h)[:CFG['N_SPEC_HARD']]
Ys_h = np.concatenate(Ys_h)[:CFG['N_SPEC_HARD']]
Xs = np.concatenate([pad_mnist(mn_tr_x), Xs_h])
Ys = np.concatenate([mn_tr_y, Ys_h])
print('Data spesialis:', Xs.shape, '| Data tumpukan train:', Xg_tr.shape)
print(f'Kanvas {CANVAS}x{CANVAS}, digit {DS}x{DS}. Rata-rata overlap terbesar per gambar: {mean_max_overlap(Pg_tr[:2000]):.3f}')

def as_dev(a): return torch.from_numpy(np.ascontiguousarray(a)).to(dev)
Xs_t, Ys_t = as_dev(Xs), as_dev(Ys)
Xg_tr_t, Xg_val_t, Xg_te_t = as_dev(Xg_tr), as_dev(Xg_val), as_dev(Xg_te)

fig, ax = plt.subplots(1, 5, figsize=(12, 3))
for i in range(5):
    ax[i].imshow(Xg_tr[i], cmap='gray'); ax[i].axis('off')
    ax[i].set_title(', '.join(CLASS_NAMES[c] for c in np.where(multihot(Yg_tr[i:i+1])[0])[0]), fontsize=7)
plt.tight_layout(); plt.show()

## 3. Model: Spesialis Biner (2 output) & Generalis (10 head)

In [ ]:
class BinarySpecialistNet(nn.Module):
    # 2 output: index 0 = BUKAN kelas dia, index 1 = kelas dia
    def __init__(self, arch='resnet18'):
        super().__init__()
        bb = getattr(models, arch)(weights=None)
        bb.conv1 = nn.Conv2d(1, 64, 3, 1, 1, bias=False)
        bb.maxpool = nn.Identity()
        bb.fc = nn.Linear(bb.fc.in_features, 2)
        self.backbone = bb
    def forward(self, x):
        return self.backbone(x)

class MTLNet(nn.Module):
    # Backbone bersama + 10 head (1 head per kelas), tiap head menghasilkan 1 logit "kelas ini ada"
    def __init__(self, arch='resnet50', n_tasks=N_CLASSES, p_drop=CFG['DROPOUT']):
        super().__init__()
        bb = getattr(models, arch)(weights=None)
        bb.conv1 = nn.Conv2d(1, 64, 3, 1, 1, bias=False)
        bb.maxpool = nn.Identity()
        feat = bb.fc.in_features
        bb.fc = nn.Identity()
        self.backbone = bb
        self.drop = nn.Dropout(p_drop)
        self.heads = nn.ModuleList([nn.Linear(feat, 1) for _ in range(n_tasks)])
    def forward(self, x):
        f = self.drop(self.backbone(x))
        return torch.cat([h(f) for h in self.heads], dim=1)   # (B, 10) logit

def prep(xb, shift=0):
    x = xb.float().div(255).unsqueeze(1)
    if shift:
        sy, sx = random.randint(-shift, shift), random.randint(-shift, shift)
        x = F.pad(x, (shift,)*4)
        x = x[:, :, shift+sy:shift+sy+CANVAS, shift+sx:shift+sx+CANVAS]
    return x

## 4. Optimizer Multi-Task (LS, PCGrad, MGDA, FAMO)

In [ ]:
def flat_grads(losses, params):
    gs = []
    for l in losses:
        g = torch.autograd.grad(l, params, retain_graph=True, allow_unused=True)
        gs.append(torch.cat([(gi if gi is not None else torch.zeros_like(p)).flatten() for gi, p in zip(g, params)]))
    return torch.stack(gs)

def pcgrad(G):
    T = len(G); out = G.clone()
    for i in range(T):
        for j in random.sample(range(T), T):
            if j == i: continue
            dot = (out[i] * G[j]).sum()
            if dot < 0:
                out[i] = out[i] - dot / (G[j].norm() ** 2 + 1e-12) * G[j]
    return out.sum(0)

def mgda_weights(G, iters=50):
    M = G @ G.T; T = len(G)
    a = torch.ones(T, device=G.device) / T
    for _ in range(iters):
        i = torch.argmin(M @ a)
        e = torch.zeros_like(a); e[i] = 1
        d = e - a
        gamma = torch.clamp(-(d @ M @ a) / (d @ M @ d + 1e-12), 0, 1)
        a = a + gamma * d
    return a

class FAMO:
    def __init__(self, T, lr=0.025, gamma=1e-3):
        self.w = torch.zeros(T, device=dev, requires_grad=True)
        self.opt = torch.optim.Adam([self.w], lr=lr, weight_decay=gamma)
        self.eps = 1e-8
    def weighted(self, losses):
        D = losses + self.eps
        z = F.softmax(self.w, -1)
        c = (z / D).sum().detach()
        return (D.log() * z / c).sum()
    def update(self, prev, curr):
        delta = (prev + self.eps).log() - (curr + self.eps).log()
        with torch.enable_grad():
            d = torch.autograd.grad(F.softmax(self.w, -1), self.w, grad_outputs=delta.detach())[0]
        self.opt.zero_grad(); self.w.grad = d; self.opt.step()

def train_mtl(model, loss_fn, n_train, T, method, epochs, lr=CFG['LR'], bs=CFG['BS']):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps = epochs * (n_train // bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=steps)
    shared = list(model.backbone.parameters())
    famo = FAMO(T) if method == 'famo' else None
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n_train, device=dev)
        tot, nb = 0.0, 0
        for s in range(0, n_train - bs + 1, bs):
            idx = perm[s:s+bs]
            losses = loss_fn(model, idx)
            opt.zero_grad()
            if method == 'ls':
                (sum(losses) / T).backward()
            elif method == 'famo':
                famo.weighted(torch.stack(losses)).backward()
            else:
                G = flat_grads(losses, shared)
                (sum(losses) / T).backward()            # grad untuk head
                g = pcgrad(G) / T if method == 'pcgrad' else (mgda_weights(G) @ G)
                o = 0
                for p in shared:
                    n = p.numel()
                    p.grad = g[o:o+n].view_as(p).clone(); o += n
            opt.step(); sched.step()
            if famo is not None:
                with torch.no_grad():
                    new = torch.stack(loss_fn(model, idx))
                famo.update(torch.stack(losses).detach(), new)
            tot += float(sum(losses).item()) / T; nb += 1
        print(f'  epoch {ep+1}/{epochs}  mean task loss = {tot/nb:.4f}')
    return model

## 5. Stage 1: Latih 10 Spesialis Biner (label asli, bobot 0.9 / 0.1)

Dengan bobot kelas `[0.1, 0.9]`, posterior spesialis bergeser sebesar `log(0.9/0.1)` ke arah "kelas dia". Saat inferensi, pergeseran ini dikoreksi (`PRIOR_CORRECT`) supaya probabilitas yang diberikan ke generalis terkalibrasi.

In [ ]:
def train_specialist(c, epochs=CFG['SPEC_EPOCHS'], lr=CFG['LR'], bs=CFG['BS']):
    set_seed(CFG['SEED'] + c)
    model = BinarySpecialistNet('resnet18').to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    n = len(Xs_t)
    steps = epochs * math.ceil(n / bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=steps)
    w = torch.tensor([CFG['W_NEG'], CFG['W_POS']], device=dev)
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n, device=dev)
        for s in range(0, n, bs):
            idx = perm[s:s+bs]
            x = prep(Xs_t[idx], shift=2)
            y = (Ys_t[idx] == c).long()                  # label asli: 1 = kelas dia, 0 = bukan
            loss = F.cross_entropy(model(x), y, weight=w)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return model

@torch.no_grad()
def spec_prob(model, X, bs=512):
    # probabilitas "kelas dia" (terkalibrasi bila PRIOR_CORRECT)
    model.eval()
    out = []
    shift = math.log(CFG['W_POS'] / CFG['W_NEG']) if CFG['PRIOR_CORRECT'] else 0.0
    for s in range(0, len(X), bs):
        z = model(prep(X[s:s+bs]))
        out.append(torch.sigmoid(z[:, 1] - z[:, 0] - shift).cpu())
    return torch.cat(out).numpy()

specialists = []
for c in range(N_CLASSES):
    t0 = time.time()
    specialists.append(train_specialist(c))
    print(f'Spesialis {c} ({CLASS_NAMES[c]}) selesai ({time.time()-t0:.0f}s)')

# Evaluasi spesialis pada MNIST test (gambar tunggal)
Xte_single = as_dev(pad_mnist(mn_te_x))
print(f'{"kelas":12s} acc    recall specificity')
for c in range(N_CLASSES):
    p = spec_prob(specialists[c], Xte_single) > 0.5
    y = mn_te_y == c
    print(f'{CLASS_NAMES[c]:12s} {(p==y).mean():.4f} {p[y].mean():.4f} {(~p[~y]).mean():.4f}')

## 6. Stage 2: Soft Target dari Spesialis untuk Tumpukan Gambar

Untuk tiap tumpukan, setiap slot di-crop lalu dinilai 10 spesialis. Target tiap head = probabilitas tertinggi kelas itu di antara slot (kelas itu ada di tumpukan). Generalis tidak melihat label asli.

In [ ]:
def teacher_presence(X, P):
    n = len(X)
    out = np.zeros((n, N_CLASSES), np.float32)
    for k in range(K_MAX):
        valid = P[:, k, 0] != -1
        crops = as_dev(crop_pad(X, P, k))
        probs = np.stack([spec_prob(specialists[c], crops) for c in range(N_CLASSES)], axis=1)
        probs[~valid] = 0.0
        out = np.maximum(out, probs)
    return torch.from_numpy(out).to(dev)

soft_t = teacher_presence(Xg_tr, Pg_tr)          # (N, 10)

# sanity check (hanya laporan, tidak dipakai untuk training)
Mh_tr = multihot(Yg_tr)
agree = ((soft_t.cpu().numpy() > THR) == Mh_tr).mean()
print(f'Kesepakatan teacher vs ground truth (threshold {THR}): {agree:.4f}')

## 7. Stage 3: Latih Generalis 10 Head (hanya dari spesialis)

In [ ]:
def gen_loss_fn(model, idx):
    logits = model(prep(Xg_tr_t[idx], shift=1))
    tgt = soft_t[idx]
    return [F.binary_cross_entropy_with_logits(logits[:, c], tgt[:, c]) for c in range(N_CLASSES)]

set_seed(CFG['SEED'])
gen = MTLNet(CFG['GEN_ARCH']).to(dev)
t0 = time.time()
train_mtl(gen, gen_loss_fn, len(Xg_tr_t), N_CLASSES, CFG['METHOD'], CFG['GEN_EPOCHS'])
print(f'Generalis selesai ({time.time()-t0:.0f}s)')

## 8. Inferensi: Threshold 1/10 + Confidence Interval 95% + Expand

* Satu sampel = satu forward pass dengan dropout aktif (BatchNorm tetap eval) + random shift.
* Untuk tiap kelas: rata-rata sampel `m`, SEM `s/√n`, batas bawah CI 95% `m - 1.96·SEM`. Kelas dianggap **ada** jika batas bawah **> 1/10**.
* **Expand**: jumlah sampel digandakan. Per gambar, jika jumlah kelas berubah dibanding ronde sebelumnya, expand lagi; jika sama, berhenti (maks `MAX_ROUNDS`).

In [ ]:
@torch.no_grad()
def sample_probs(model, Xb):
    model.eval()
    model.drop.train()                    # MC-dropout aktif, BatchNorm tetap eval
    return torch.sigmoid(model(prep(Xb, shift=1))).cpu()

@torch.no_grad()
def ci_expand_predict(model, X, n0=CFG['N0'], max_rounds=CFG['MAX_ROUNDS'],
                      z=CFG['CI_Z'], thr=THR, bs=256):
    N = len(X)
    s1 = torch.zeros(N, N_CLASSES); s2 = torch.zeros(N, N_CLASSES)
    n = torch.zeros(N)

    def add(idx, m):
        for s in range(0, len(idx), bs):
            b = idx[s:s+bs]
            Xb = X[b.to(X.device)]
            for _ in range(m):
                p = sample_probs(model, Xb)
                s1[b] += p; s2[b] += p * p
            n[b] += m

    def lower_bound(idx):
        nn_ = n[idx].unsqueeze(1)
        mean = s1[idx] / nn_
        var = ((s2[idx] - nn_ * mean ** 2) / (nn_ - 1)).clamp(min=0)
        return mean, mean - z * (var / nn_).sqrt()

    allidx = torch.arange(N)
    add(allidx, n0)
    mean, lo = lower_bound(allidx)
    present = lo > thr
    count = present.sum(1)
    rounds = torch.ones(N, dtype=torch.long)
    active = allidx.clone()
    # riwayat per ronde (gambar yang sudah berhenti membawa nilai finalnya)
    hist = dict(present=[present.clone()], count=[count.clone()], lower=[lo.clone()],
                mean=[mean.clone()], n_total=[n0], n_active=[N])

    for r in range(2, max_rounds + 2):
        if len(active) == 0: break
        n_active_now = len(active)
        add(active, int(n[active[0]].item()))          # gandakan jumlah sampel
        m_a, lo_a = lower_bound(active)
        pres_a = lo_a > thr
        cnt_a = pres_a.sum(1)
        changed = cnt_a != count[active]
        mean[active], lo[active], present[active], count[active] = m_a, lo_a, pres_a, cnt_a
        rounds[active] = r
        hist['present'].append(present.clone()); hist['count'].append(count.clone())
        hist['lower'].append(lo.clone()); hist['mean'].append(mean.clone())
        hist['n_total'].append(int(n[active[0]].item())); hist['n_active'].append(n_active_now)
        active = active[changed]                       # yang berubah di-expand lagi
    return dict(mean=mean, lower=lo, present=present, count=count, rounds=rounds, n=n,
                unstable=len(active), hist=hist)

def evaluate_sets(pred, truth):
    pred, truth = np.asarray(pred, bool), np.asarray(truth, bool)
    tp = (pred & truth).sum(); fp = (pred & ~truth).sum(); fn = (~pred & truth).sum()
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    return dict(exact_set=(pred == truth).all(1).mean(),
                count_acc=(pred.sum(1) == truth.sum(1)).mean(),
                precision=prec, recall=rec, f1=2*prec*rec/max(prec+rec, 1e-9))

# --- Evaluasi pada test set ---
t0 = time.time()
res = ci_expand_predict(gen, Xg_te_t)
print(f'Inferensi CI + expand selesai ({time.time()-t0:.0f}s). '
      f'Rata-rata ronde: {res["rounds"].float().mean():.2f}, belum stabil di ronde maks: {res["unstable"]}')

base = evaluate_sets((res['mean'] > THR).numpy(), Mh_te)      # threshold 1/10 saja, tanpa CI
ci   = evaluate_sets(res['present'].numpy(), Mh_te)           # CI 95% + expand
print('Threshold 1/10 saja      :', {k: round(float(v), 4) for k, v in base.items()})
print('CI 95% + expand (final)  :', {k: round(float(v), 4) for k, v in ci.items()})

## 9. Analisis Sebelum vs Sesudah Expand

* **Ronde 1** = hasil sebelum expand (hanya `N0` sampel). **Ronde terakhir** = hasil setelah expand.
* Tabel per ronde, tabel transisi jumlah kelas, dan beberapa contoh gambar untuk presentasi.

In [ ]:
H = res['hist']; R = len(H['count'])
true_cnt = torch.from_numpy(Mh_te.sum(1)).long()

# --- 9a. Tabel metrik per ronde ---
print('=== Metrik per ronde expand (test set) ===')
print(f'{"ronde":>5} {"n sampel":>9} {"gbr di-expand":>14} {"gbr berubah":>12} {"rata2 jml kelas":>16} {"count_acc":>10} {"exact_set":>10} {"F1":>7}')
for r in range(R):
    m = evaluate_sets(H['present'][r].numpy(), Mh_te)
    chg = '-' if r == 0 else int((H['count'][r] != H['count'][r-1]).sum())
    print(f'{r+1:>5} {H["n_total"][r]:>9} {H["n_active"][r]:>14} {str(chg):>12} '
          f'{H["count"][r].float().mean():>16.3f} {m["count_acc"]:>10.4f} {m["exact_set"]:>10.4f} {m["f1"]:>7.4f}')
print(f'(rata-rata jumlah kelas sebenarnya: {true_cnt.float().mean():.3f})')

# --- 9b. Transisi jumlah kelas: sebelum (ronde 1) -> sesudah (final) ---
c0, cf = H['count'][0], H['count'][-1]
print()
print('=== Transisi jumlah kelas terprediksi: sebelum expand -> sesudah expand ===')
vals = sorted(set(c0.tolist()) | set(cf.tolist()))
print('sebelum / sesudah ' + ''.join(f'{v:>7}' for v in vals))
for a in vals:
    print(f'{a:>16} ' + ''.join(f'{int(((c0 == a) & (cf == b)).sum()):>7}' for b in vals))

err0, errf = (c0 - true_cnt).abs(), (cf - true_cnt).abs()
print()
print(f'Gambar jumlahnya berubah setelah expand : {int((c0 != cf).sum())} dari {len(c0)}')
print(f'  membaik  (selisih ke jumlah benar mengecil): {int((errf < err0).sum())}')
print(f'  memburuk (selisih membesar)                 : {int((errf > err0).sum())}')
print(f'  berubah tapi selisih sama                   : {int(((c0 != cf) & (errf == err0)).sum())}')
print(f'Rata-rata |jumlah prediksi - jumlah benar|: sebelum {err0.float().mean():.3f} -> sesudah {errf.float().mean():.3f}')

# --- 9c. Jejak beberapa contoh ---
def show_trace(i):
    true = [CLASS_NAMES[c] for c in np.where(Mh_te[i])[0]]
    print(f'Gambar #{i} | benar ({len(true)}): {", ".join(true)}')
    for r in range(R):
        names = [CLASS_NAMES[c] for c in np.where(H['present'][r][i].numpy())[0]]
        tag = ''
        if r > 0 and H['count'][r][i] == H['count'][r-1][i] and r == int(res['rounds'][i]) - 1:
            tag = '  <- sama dengan ronde sebelumnya, BERHENTI'
        print(f'   ronde {r+1} (n={H["n_total"][r]:>3}): {int(H["count"][r][i])} kelas -> {", ".join(names)}{tag}')
        if int(res['rounds'][i]) == r + 1:
            break
    # kelas yang berubah status
    for c in range(N_CLASSES):
        a, b = bool(H['present'][0][i, c]), bool(H['present'][-1][i, c])
        if a != b:
            print(f'   * {CLASS_NAMES[c]}: batas bawah CI {H["lower"][0][i, c]:.3f} -> {H["lower"][-1][i, c]:.3f} '
                  f'(threshold {THR}) => {"DITAMBAH" if b else "DIBUANG"}')
    print()

improved = ((err0 > 0) & (errf == 0)).nonzero().flatten().tolist()
worsened = ((err0 == 0) & (errf > 0)).nonzero().flatten().tolist()
changed_idx = (c0 != cf).nonzero().flatten().tolist()
stable = ((res['rounds'] == 2) & (c0 == cf) & (cf == true_cnt)).nonzero().flatten().tolist()
pick = improved[:3] + worsened[:1]
pick += [i for i in changed_idx if i not in pick][:max(0, 4 - len(pick))]
print('=== Contoh jejak expand ===')
print('-- gambar yang jumlah kelasnya BERUBAH --')
for i in pick: show_trace(i)
print('-- gambar yang STABIL (tidak berubah setelah expand pertama) --')
for i in stable[:1]: show_trace(i)

# --- 9d. Grafik ---
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
rs = np.arange(1, R + 1)
accs = [evaluate_sets(H['present'][r].numpy(), Mh_te) for r in range(R)]
ax[0].plot(rs, [a['count_acc'] for a in accs], 'o-', label='count_acc')
ax[0].plot(rs, [a['exact_set'] for a in accs], 's-', label='exact_set')
ax[0].plot(rs, [a['f1'] for a in accs], '^-', label='F1')
ax[0].set_xticks(rs); ax[0].set_xlabel('ronde expand (1 = sebelum expand)'); ax[0].set_title('Metrik per ronde'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].bar(rs, H['n_active']); ax[1].set_xticks(rs)
ax[1].set_xlabel('ronde'); ax[1].set_ylabel('jumlah gambar'); ax[1].set_title('Gambar yang diproses tiap ronde')
for r_, v in zip(rs, H['n_active']): ax[1].text(r_, v, str(v), ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.show()

shown = (pick + stable[:1])[:5]
if shown:
    fig, ax = plt.subplots(1, len(shown), figsize=(3.4 * len(shown), 4.2), squeeze=False)
    for a, i in zip(ax[0], shown):
        a.imshow(Xg_te[i], cmap='gray'); a.axis('off')
        t = [CLASS_NAMES[c] for c in np.where(Mh_te[i])[0]]
        b = [CLASS_NAMES[c] for c in np.where(H['present'][0][i].numpy())[0]]
        f = [CLASS_NAMES[c] for c in np.where(H['present'][-1][i].numpy())[0]]
        a.set_title(f'Benar ({len(t)}): {", ".join(t)}\nSebelum ({len(b)}): {", ".join(b)}\nSesudah ({len(f)}): {", ".join(f)}', fontsize=7)
    plt.tight_layout(); plt.show()

## 10. Visualisasi Contoh

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(16, 7))
for a, i in zip(ax.ravel(), range(8)):
    a.imshow(Xg_te[i], cmap='gray'); a.axis('off')
    true = [CLASS_NAMES[c] for c in np.where(Mh_te[i])[0]]
    pred = [CLASS_NAMES[c] for c in np.where(res['present'][i].numpy())[0]]
    a.set_title(f'Benar ({len(true)}): {", ".join(true)}\n'
                f'Prediksi ({int(res["count"][i])}): {", ".join(pred)}\n'
                f'ronde={int(res["rounds"][i])}, n={int(res["n"][i])}', fontsize=8)
plt.tight_layout(); plt.show()

# CI per kelas untuk 1 gambar
i = 0
m, lo = res['mean'][i].numpy(), res['lower'][i].numpy()
plt.figure(figsize=(8, 3.5))
plt.bar(range(N_CLASSES), m, yerr=m - lo, capsize=3, color=['tab:green' if p else 'tab:gray' for p in res['present'][i].numpy()])
plt.axhline(THR, color='r', ls='--', label='ekspektasi 1/10')
plt.xticks(range(N_CLASSES), CLASS_NAMES, rotation=45, ha='right'); plt.ylabel('P(kelas ada)')
plt.legend(); plt.tight_layout(); plt.show()